# Phase 8: secondary metrics for the appendix (half A, no GPU)

**CRISP-DM stage:** Evaluation

The headline metrics are fixed by the benchmark. These secondary metrics go in the appendix, as the blueprint asks. They are calculated from the photo scores every run already saved, so no model is run again.

- **Image AUPRC (average precision).** Sort photos from most to least suspicious. At each defective photo, note the precision so far, then average. 1.0 means every defective photo comes first. **Careful:** a model that guesses at random already scores about the share of defective photos, 0.78 for Sheet Metal and 0.70 for Vial on half A, because most half A photos are defective. So AUPRC must be read against that baseline, not against 0.
- **Precision and recall at a line.** Precision: of the rejected photos, the share that are defective. Recall: of the defective photos, the share that are rejected. Shown for the benchmark line, and for the cost-based line (cost ratio 10, defect rate 1 percent) from Phase 7.
- **Pixel average precision** needs the heat maps, which were not saved. It is calculated in the resolution-test run and in the Final test, not here.

**Limits:** the cost-based line was chosen on the same half A photos, so its precision and recall are optimistic. PatchCore's numbers are averaged over its 3 seeds.

**Half B is not read.**

**Runs on:** the laptop.

In [ ]:
import csv
import json
import sys
from pathlib import Path

import torch

# Finds the project folder, whether the notebook runs from the project or from notebooks/
PROJECT_FOLDER = Path.cwd()
if not (PROJECT_FOLDER / "src").exists():
    PROJECT_FOLDER = PROJECT_FOLDER.parent
sys.path.insert(0, str(PROJECT_FOLDER))
from src import evaluate

RESULT_FOLDERS = [PROJECT_FOLDER / "outputs/results/05_core", PROJECT_FOLDER / "outputs/results/07_core"]
COST_FILE = PROJECT_FOLDER / "outputs/results/07_evaluation/cost_by_run.csv"
OUTPUT_FOLDER = PROJECT_FOLDER / "outputs/results/08_secondary"
OUTPUT_FOLDER.mkdir(parents=True, exist_ok=True)
PRODUCTS = ["sheet_metal", "vial"]
CONFIGS = ["C1", "C2", "C3", "C4", "C5", "C6", "C7", "C8"]
# The project's main business setting, the same as in Phases 6 and 7
DEFECT_RATE = "0.01"
COST_RATIO = "10"


def read_scores(score_file):
    """Return labels (True for defective) and scores from one run's file."""
    labels = []
    scores = []
    with open(score_file, encoding="utf-8") as file:
        for row in csv.DictReader(file):
            labels.append(row["label"] == "bad")
            scores.append(float(row["score"]))
    return torch.tensor(labels), torch.tensor(scores)


# The cost-based line of every run, at the main business setting
cost_lines = {}
with open(COST_FILE, encoding="utf-8") as file:
    for row in csv.DictReader(file):
        if row["defect_rate"] == DEFECT_RATE and row["cost_ratio"] == COST_RATIO:
            cost_lines[(row["config"], row["product"], row["seed"])] = float(row["cheapest_threshold"])
print(len(cost_lines), "cost-based lines read")

## 1. Calculate the metrics for every run, then average over seeds

In [ ]:
def average(values):
    """Average a list of numbers, or return "not defined" if any value is not defined."""
    for value in values:
        if value == "not defined":
            return "not defined"
    return round(sum(values) / len(values), 4)


rows = []
for config_id in CONFIGS:
    for product in PRODUCTS:
        score_files = []
        for folder in RESULT_FOLDERS:
            score_files.extend(sorted((folder / config_id).glob(config_id + "_" + product + "_seed*_photo_scores.csv")))
        measures = {"image_auprc": [], "benchmark_precision": [], "benchmark_recall": [],
                    "cost_line_precision": [], "cost_line_recall": []}
        for score_file in score_files:
            seed = score_file.name.split("_seed")[1].split("_")[0]
            labels, scores = read_scores(score_file)
            metrics_file = score_file.parent / score_file.name.replace("_photo_scores.csv", "_metrics.json")
            with open(metrics_file, encoding="utf-8") as file:
                benchmark_line = json.load(file)["threshold"]
            measures["image_auprc"].append(evaluate.image_auprc(scores, labels))
            at_benchmark = evaluate.precision_recall_at(scores, labels, benchmark_line)
            measures["benchmark_precision"].append(at_benchmark["precision"])
            measures["benchmark_recall"].append(at_benchmark["recall"])
            at_cost_line = evaluate.precision_recall_at(scores, labels, cost_lines[(config_id, product, seed)])
            measures["cost_line_precision"].append(at_cost_line["precision"])
            measures["cost_line_recall"].append(at_cost_line["recall"])
        row = {"config": config_id, "product": product, "seeds": len(score_files),
               "random_guess_auprc": round(float(labels.float().mean()), 4)}
        for name in measures:
            row[name] = average(measures[name])
        rows.append(row)

with open(OUTPUT_FOLDER / "secondary_metrics_half_a.csv", "w", newline="") as file:
    writer = csv.DictWriter(file, fieldnames=list(rows[0].keys()))
    writer.writeheader()
    for row in rows:
        writer.writerow(row)

for product in PRODUCTS:
    print("===", product)
    print("config  AUPRC (random)   benchmark line: precision, recall   cost line: precision, recall")
    for row in rows:
        if row["product"] == product:
            print(row["config"].ljust(7), str(row["image_auprc"]).ljust(7), "(" + str(row["random_guess_auprc"]) + ")   ",
                  str(row["benchmark_precision"]).ljust(8), str(row["benchmark_recall"]).ljust(27),
                  str(row["cost_line_precision"]).ljust(8), row["cost_line_recall"])
print("Saved", OUTPUT_FOLDER / "secondary_metrics_half_a.csv")